# Guia Completo de Manipulação de Dados com Pandas
### Referência Prática Organizada em Três Níveis de Complexidade

---

## Sumário

| Módulo | Nível | Temas |
|--------|-------|-------|
| **1** | Básico | Diagnóstico, Seleção, Limpeza, Métricas |
| **2** | Intermediário | Joins, Agrupamento, Séries Temporais, Strings |
| **3** | Avançado | Performance, Pipelines, Estruturas Complexas |

---

## Configuração do Ambiente

Execute a célula abaixo antes dos exemplos: ela **importa as bibliotecas** e **carrega os arquivos CSV** utilizados no tutorial. Todos os arquivos deverão estar no mesmo diretório do notebook.

In [ ]:
import ast
import json

import numpy as np
import pandas as pd

# DataFrame principal (df) usado na maioria dos exemplos
df = pd.read_csv('vendas.csv', sep=';', encoding='utf-8')

# O CSV lê tudo como texto: converte a coluna de listas (usada no exemplo de .explode())
df['tags_produtos'] = df['tags_produtos'].apply(ast.literal_eval)

# data_hora é convertida para datetime na seção de Séries Temporais (Módulo 2)
print(f'df carregado: {df.shape[0]} linhas x {df.shape[1]} colunas')

# MÓDULO 1 — Nível Básico
## Diagnóstico, Inspeção e Manipulação Inicial de Dados

---
## 1.1 Diagnóstico e Inspeção Estrutural

### `.head()` / `.tail()`

Retorna as **primeiras** ou **últimas** *N* linhas do DataFrame para inspeção visual rápida.

**Sintaxe:** `df.head(n=5)`

In [ ]:
df.head(3)  # Exibe as 3 primeiras linhas

### `.info()`

Exibe o **resumo técnico** do DataFrame: quantidade de linhas/colunas, tipos de dados de cada variável (`dtypes`) e uso de memória.

**Sintaxe:** `df.info()`

In [ ]:
df.info()

### `.describe()`

Gera **estatísticas descritivas** básicas para colunas numéricas: contagem, média, desvio padrão, quartis, mínimo e máximo. Perceba que as colunas com valores nulos (`valor`, `valor_venda`) aparecem com contagem menor.

**Sintaxe:** `df.describe()`

In [ ]:
df.describe()

### `.shape`

Atributo que retorna uma **tupla** indicando o número de linhas e colunas no formato `(linhas, colunas)`.

**Sintaxe:** `df.shape`

In [ ]:
linhas, colunas = df.shape
print(f'Dimensões: {linhas} linhas x {colunas} colunas')

---
## 1.2 Seleção e Fatiamento

### `.loc[]`

Seleciona dados baseando-se em **rótulos** de linhas e nomes das colunas. Aceita condições booleanas.

**Sintaxe:** `df.loc[linhas_rótulo_ou_condição, colunas_rótulo]`

In [ ]:
df.loc[df['valor'] > 100, ['cliente_id', 'valor']]

### `.iloc[]`

Seleciona dados através de seus **índices inteiros posicionais** (base zero).

**Sintaxe:** `df.iloc[inicio_linha:fim_linha, inicio_coluna:fim_coluna]`

In [ ]:
df.iloc[0:5, 0:2]  # Primeiras 5 linhas e primeiras 2 colunas

### `.isin()`

Filtra registros cujos valores de uma coluna estejam **presentes em uma lista** especificada.

**Sintaxe:** `df['coluna'].isin([valor1, valor2])`

In [ ]:
df[df['estado'].isin(['SP', 'RJ'])]

---
## 1.3 Limpeza e Tratamento

### `.dropna()`

Remove linhas ou colunas que contenham **valores nulos** (`NaN`). No arquivo, 2 linhas de `valor_venda` estão vazias — elas serão removidas.

**Sintaxe:** `df.dropna(subset=['coluna_chave'], how='any', inplace=False)`

In [ ]:
df_limpo = df.dropna(subset=['valor_venda'])
df_limpo.shape

### `.fillna()`

Preenche **valores nulos** (`NaN`) por um valor fixo ou calculado (média, mediana, etc.). Aqui preenchemos `valor` com a mediana.

**Sintaxe:** `df['coluna'].fillna(value)`

In [ ]:
df['valor'] = df['valor'].fillna(df['valor'].median())
df['valor'].isna().sum()  # 0 = sem valores nulos restantes

### `.drop()`

Elimina **colunas ou linhas** especificadas do DataFrame.

**Sintaxe:** `df.drop(columns=['col1', 'col2'])`

In [ ]:
df_reduzido = df.drop(columns=['rascunho', 'temp'])
df_reduzido.shape

### `.rename()`

Renomeia os **rótulos** de colunas ou índices.

**Sintaxe:** `df.rename(columns={'nome_antigo': 'nome_novo'})`

In [ ]:
df = df.rename(columns={'val': 'valor_venda', 'cli': 'cliente_id'})
list(df.columns)

---
## 1.4 Métricas e Agregações Simples

### `.value_counts()`

Retorna a **contagem de frequência** de cada valor único presente em uma coluna.

**Sintaxe:** `df['coluna'].value_counts(normalize=False)`

In [ ]:
df['categoria'].value_counts()

### `.nunique()`

Conta o número de **elementos distintos** (únicos) em uma coluna ou DataFrame.

**Sintaxe:** `df['coluna'].nunique()`

In [ ]:
total_clientes_unicos = df['cliente_id'].nunique()
print(f'Quantidade de clientes únicos: {total_clientes_unicos}')

---
# MÓDULO 2 — Nível Intermediário
## Transformação, Relacionamentos e Séries Temporais

---
## 2.1 Cruzamento de Dados (Joins)

**Preparação — `pd.merge()`:** carrega o DataFrame de vendas (cópia do principal) e a tabela de clientes. A junção cruza as duas tabelas pela chave `cliente_id`.

In [ ]:
vendas = df.copy()
clientes = pd.read_csv('clientes.csv', sep=';', encoding='utf-8')

print(clientes.head(3))

### `pd.merge()`

Realiza a **junção relacional** (estilo SQL) entre dois DataFrames com base em chaves em comum.

**Sintaxe:** `pd.merge(left, right, on='chave', how='inner'|'left'|'right'|'outer')`

In [ ]:
df_completo = pd.merge(vendas, clientes, on='cliente_id', how='left')
df_completo
df_completo.shape  # 20 vendas + colunas de cliente

**Preparação — `pd.concat()`:** carrega os arquivos dos dois semestres (mesma estrutura de colunas de `vendas.csv`, condição para o empilhamento).

In [ ]:
df_semestre1 = pd.read_csv('vendas_semestre1.csv', sep=';', encoding='utf-8')
df_semestre2 = pd.read_csv('vendas_semestre2.csv', sep=';', encoding='utf-8')

print('Semestre 1:', df_semestre1.shape, '| Semestre 2:', df_semestre2.shape)

### `pd.concat()`

**Empilha** DataFrames ao longo de um eixo (verticalmente adicionando linhas ou horizontalmente adicionando colunas).

**Sintaxe:** `pd.concat([df1, df2], axis=0, ignore_index=True)`

In [ ]:
df_ano_completo = pd.concat([df_semestre1, df_semestre2], axis=0, ignore_index=True)
df_ano_completo.shape  # 3 + 3 linhas

**Preparação — `.join()`:** carrega `df1` e `df2` usando a coluna `id` como índice — o `.join()` combina pelos índices de linha.

In [ ]:
df1 = pd.read_csv('df1.csv', sep=';', encoding='utf-8').set_index('id')
df2 = pd.read_csv('df2.csv', sep=';', encoding='utf-8').set_index('id')

print(df1)
print(df2)

### `.join()`

Combina dois DataFrames com base em seus **índices de linha** em vez de colunas explícitas.

**Sintaxe:** `df1.join(df2, how='left')`

In [ ]:
df1.join(df2)  # sem lsuffix/rsuffix aqui porque as colunas não colidem

---
## 2.2 Agrupamento e Reestruturação

### `.groupby()` e `.agg()`

Agrupa dados por uma ou mais **categorias** e aplica funções de agregação (soma, média, contagem) sobre as variáveis.

**Sintaxe:** `df.groupby('coluna_grupo').agg({'coluna_valor': 'função'})`

In [ ]:
df.groupby('regiao').agg({'valor_venda': ['sum', 'mean']})

### `.transform()`

Aplica uma função de grupo e retorna um objeto com a **mesma quantidade de linhas** do DataFrame original (ideal para calcular percentuais do total do grupo).

**Sintaxe:** `df.groupby('grupo')['coluna'].transform('funcao')`

In [ ]:
df['media_categoria'] = df.groupby('categoria')['valor'].transform('mean')
df[['categoria', 'valor', 'media_categoria']].head(5)

### `pd.pivot_table()`

Constrói uma **tabela dinâmica** relacionando índices, colunas e uma função agregadora sobre valores.

**Sintaxe:** `pd.pivot_table(df, values='v', index='i', columns='c', aggfunc='sum')`

In [ ]:
pd.pivot_table(df, values='valor', index='categoria', columns='estado', aggfunc='sum', fill_value=0)

**Observação — `pd.melt()`:** o exemplo opera sobre a tabela **wide** carregada de `vendas_mensal.csv` (clientes em linhas, meses `jan/fev/mar` em colunas).

In [ ]:
mensal = pd.read_csv('vendas_mensal.csv', sep=';', encoding='utf-8')
mensal.head()

### `pd.melt()`

Transforma o DataFrame do formato *wide* (largo) para o formato *long* (comprido), unificando várias colunas em pares de **variável-valor**.

**Sintaxe:** `pd.melt(df, id_vars=['id'], value_vars=['col1', 'col2'], var_name='var', value_name='val')`

In [ ]:
pd.melt(mensal, id_vars=['cliente_id'], value_vars=['jan', 'fev', 'mar'], var_name='mes', value_name='venda').head(8)

### `.unstack()`

Desempilha um nível dos **índices rotulados** (`MultiIndex`) para as colunas do DataFrame.

**Sintaxe:** `df.unstack(level=-1)`

In [ ]:
df.groupby(['regiao', 'categoria'])['valor'].sum().unstack()

---
## 2.3 Séries Temporais

### `pd.to_datetime()`

Converte textos ou números em objetos no formato de **data e hora** nativo do pandas (`datetime64`).

**Sintaxe:** `pd.to_datetime(arg, format=None)`

In [ ]:
df['data_hora'] = pd.to_datetime(df['data_hora'], format='%Y-%m-%d %H:%M:%S')
df['data_hora']

### `.dt` (Acessor Temporal)

Dá acesso a **propriedades de data** como ano, mês, dia, dia da semana, hora, etc.

**Sintaxe:** `df['coluna_datetime'].dt.propriedade`

In [ ]:
df['ano'] = df['data_hora'].dt.year
df['dia_semana'] = df['data_hora'].dt.day_name()
df[['data_hora', 'ano', 'dia_semana']].head(5)

### `.resample()`

Agrupa dados temporais por **frequências customizadas** (diária, mensal, trimestral). A coluna `data_hora` já é `datetime` (convertida acima).

**Sintaxe:** `df.set_index('data').resample('rule').sum()`

In [ ]:
df.set_index('data_hora').resample('ME')['valor'].sum()  # Agrupamento ao fim de cada mês

### `.shift()`

Desloca os valores de uma coluna por um número determinado de **períodos** (útil para calcular variação percentual em relação ao período anterior).

**Sintaxe:** `df['coluna'].shift(periods=1)`

In [ ]:
df['venda_mes_anterior'] = df['valor_venda'].shift(1)
df[['valor_venda', 'venda_mes_anterior']].head(5)

---
## 2.4 Manipulação de Strings

### `.str.contains()` / `.str.replace()` / `.str.extract()`

Aplica **operações e expressões regulares** diretamente em colunas de texto.

**Sintaxe:** `df['coluna'].str.metodo()`

In [ ]:
gmail = df[df['email'].str.contains('@gmail.com')]
print('Emails do Gmail:', len(gmail))

df['codigo'] = df['codigo'].str.replace('-', '')

---
# MÓDULO 3 — Nível Avançado
## Performance, Pipelines e Estruturas Complexas

---
## 3.1 Otimização e Big Data

### `.astype('category')`

Converte colunas com strings repetidas no tipo **categórico**, reduzindo significativamente o consumo de memória RAM.

**Sintaxe:** `df['coluna'] = df['coluna'].astype('category')`

In [ ]:
df['estado'] = df['estado'].astype('category')
print(df['estado'].dtype)   # category
print(df['estado'].cat.categories)

### `read_csv(chunksize=N)`

Lê arquivos CSV **massivos** dividindo-os em blocos (*chunks*) iteráveis para não estourar a memória RAM.

**Sintaxe:** `pd.read_csv(file, chunksize=tamanho_lote)`

No exemplo a seguir, usamos o próprio `vendas.csv` com `chunksize=5` apenas para demonstrar o padrão (substitua pelo arquivo massivo real).

In [ ]:
def processar(chunk):
    print(f'  bloco com {chunk.shape[0]} linhas')

for chunk in pd.read_csv('vendas.csv', sep=';', chunksize=5):
    processar(chunk)

### `pd.read_parquet()` / `.to_parquet()`

Lê e grava dados no formato **colunar comprimido Parquet** (padrão em ecossistemas de Big Data). Requer `pyarrow` ou `fastparquet` instalado.

**Sintaxe:** `pd.read_parquet('arquivo.parquet')`

In [ ]:
df.to_parquet('dados_otimizados.parquet', compression='snappy')

df_parquet = pd.read_parquet('dados_otimizados.parquet')
df_parquet.shape

---
## 3.2 Vetorização de Condicionais

### `np.where()`

Operação condicional **vetorizada** similar ao comando IF do Excel.

**Sintaxe:** `np.where(condicao, valor_se_verdadeiro, valor_se_falso)`

In [ ]:
df['status_valor'] = np.where(df['valor'] > 500, 'Alto', 'Baixo')
df[['valor', 'status_valor']].head(5)

### `np.select()`

Permite criar colunas condicionais com **múltiplas regras** e resultados em código compilado C.

**Sintaxe:** `np.select(lista_condicoes, lista_escolhas, default)`

In [ ]:
condicoes = [df['valor'] >= 1000, df['valor'] >= 500]
escolhas = ['Classe A', 'Classe B']

df['categoria_cliente'] = np.select(condicoes, escolhas, default='Classe C')
df['categoria_cliente'].value_counts()

### `.query()` e `.eval()`

Executa filtragens e expressões matemáticas diretamente com **sintaxe otimizada** baseada em C e NumExpr.

**Sintaxe:** `df.query('expressão_string')`

In [ ]:
df_filtrado = df.query("valor > 500 and status == 'Concluído'")
df_filtrado.shape

---
## 3.3 Pipelines e Processamento de Dados Estruturados

### `.assign()`

Cria ou altera colunas em um DataFrame retornando um **novo objeto**, permitindo o encadeamento de métodos.

**Sintaxe:** `df.assign(nova_coluna=expressao)`

In [ ]:
df_novo = df.assign(imposto=lambda x: x['valor'] * 0.1)
df_novo[['valor', 'imposto']].head(5)

### `.pipe()`

Aplica funções customizadas encadeadas para formar um **pipeline de transformação** contínuo e modular.

**Sintaxe:** `df.pipe(funcao_customizada)`

In [ ]:
def remover_nulos(df):
    return df.dropna(subset=['valor_venda'])

df_final = df.pipe(remover_nulos)
df_final.shape

### `pd.json_normalize()`

Transforma dados no formato JSON **hierárquico/aninhado** em uma tabela plana regular.

**Sintaxe:** `pd.json_normalize(dados_json)`

In [ ]:
df_flat = pd.json_normalize(df['detalhes_json'].apply(json.loads))
df_flat.head()

### `.explode()`

Transforma cada elemento de uma **lista** presente em uma célula em uma nova linha individual no DataFrame.

**Sintaxe:** `df.explode('coluna_com_lista')`

A coluna `tags_produtos` já foi convertida de string para lista na **célula de configuração**, então o método funciona diretamente.

In [ ]:
df_expandido = df.explode('tags_produtos')
df_expandido[['cliente_id', 'tags_produtos']].head(8)

---
# Resumo Rápido de Referência

| Função | Descrição |
|--------|-----------|
| `.head(n)` | Primeiras *n* linhas |
| `.info()` | Resumo técnico do DataFrame |
| `.describe()` | Estatísticas descritivas |
| `.shape` | Dimensões (linhas, colunas) |
| `.loc[]` | Seleção por rótulo |
| `.iloc[]` | Seleção por posição |
| `.isin()` | Filtrar por lista |
| `.dropna()` | Remover nulos |
| `.fillna()` | Preencher nulos |
| `.drop()` | Remover colunas/linhas |
| `.rename()` | Renomear colunas |
| `.value_counts()` | Contagem de frequência |
| `.nunique()` | Valores únicos |
| `pd.merge()` | Junção relacional |
| `pd.concat()` | Empilhar DataFrames |
| `.groupby().agg()` | Agrupar e agregar |
| `.transform()` | Função de grupo preservando linhas |
| `pd.pivot_table()` | Tabela dinâmica |
| `pd.melt()` | Wide para long |
| `pd.to_datetime()` | Converter para datetime |
| `.dt.` | Acessor temporal |
| `.resample()` | Reamostragem temporal |
| `.shift()` | Deslocar valores |
| `.str.` | Operações em strings |
| `.astype('category')` | Otimizar memória |
| `np.where()` | Condicional vetorizada |
| `np.select()` | Múltiplas condições |
| `.query()` | Filtragem otimizada |
| `.assign()` | Criar colunas encadeadas |
| `.pipe()` | Pipeline de transformação |
| `pd.json_normalize()` | JSON para tabela plana |
| `.explode()` | Listas para linhas |